# 04 — Earth Pulse: AI scenario cards (one click, GPU)

Generates the **three scenario images** (lower / median / upper) of a place, conditioned on
the latest real NASA GIBS tile + the model's predicted NDVI delta.

**Honesty rules (enforced in code):** label is always `AI-generated scenario — not an observation`.
Geography is preserved (img2img conditioning); only land-cover balance shifts. Never a forecast photo.

## What you need
1. **Runtime → Change runtime type → GPU** (T4 free tier is fine)
2. Just press **Run all** — no API keys, no logins anywhere.

Total time ≈ 10–15 min (most of it is the ~4 GB model download).

In [ ]:
import subprocess, sys, os, json, glob, zipfile
subprocess.run(['nvidia-smi'], check=False)

## 1. Clone Earth Pulse

In [ ]:
%cd /content
if not os.path.exists('/content/earth-pulse'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/cyberhatc/earth-pulse.git'], check=True)
%cd /content/earth-pulse

## 2. Install dependencies
(CPU wheels first for diffusers bits; torch is already present on Colab GPU images)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers>=0.30', 'transformers>=4.44', 'accelerate>=0.33',
    'pillow', 'requests', 'numpy', 'scipy', 'fastapi', 'uvicorn',
    'matplotlib', 'tqdm'], check=True)
print('deps installed')

## 3. Sanity check (build demo store + forecaster + verify)

In [ ]:
subprocess.run([sys.executable, 'pipeline/run_pipeline.py', '--sample'], check=True)
subprocess.run([sys.executable, 'engine/run_engine.py'], check=True)
subprocess.run([sys.executable, 'models/train.py'], check=True)

## 4. Generate the three scenario cards for Bengaluru

- GPU → `runwayml/stable-diffusion-v1-5` @ 30 steps (best quality).
- CPU → `stabilityai/sd-turbo` @ 4 steps, guidance 0.0 (distilled — works, slower, slightly softer).


In [ ]:
import torch
gpu = torch.cuda.is_available()
print('GPU available:', gpu)
if gpu:
    print('Plan: SD 1.5 @ 30 steps (default)')
    env = dict(os.environ, EP_SCENARIO_IMAGES='1')
else:
    print('No GPU — switching to SD-Turbo (distilled, 4 steps, guidance 0.0). Slow but works on CPU.')
    env = dict(os.environ, EP_SCENARIO_IMAGES='1',
               EP_SCENARIO_MODEL='stabilityai/sd-turbo',
               EP_SCENARIO_STEPS='4',
               EP_SCENARIO_GUIDANCE='0.0')
proc = subprocess.run([sys.executable, 'models/scenario_cards.py', '--generate',
                      '--lat', '12.97', '--lon', '77.59', '--parameter', 'ndvi'],
                     capture_output=True, text=True, env=env, check=False)
print(proc.stdout[-5000:])
print(proc.stderr[-2500:])
assert 'Status' in proc.stdout or ('generated' in proc.stdout) or proc.returncode in (0,), 'generation failed — see output above'

## 5. Look at them

In [ ]:
from IPython.display import Image, display
pngs = sorted(glob.glob('/content/earth-pulse/models/artifacts/scenarios/*.png'))
print('generated:', pngs)
for p in pngs:
    display(Image(p))

In [ ]:
with zipfile.ZipFile('/content/earth-pulse-scenarios.zip', 'w') as zf:
    for p in pngs: zf.write(p, os.path.basename(p))
print('zip ready — use the Files pane (left) to download earth-pulse-scenarios.zip')

## Troubleshooting

**"await connected: disconnected" — the runtime dropped before any cell ran (your browser's link to Colab):**
1. Close the tab. Open a **private/incognito window** with **ad-block/privacy extensions OFF** (they block ssl.gstatic.com). Allow third-party cookies on *.google.com.
2. Reload the notebook URL — Colab should show a fresh 'Connect' banner → click **Connect** → **Runtime → Run all**.
3. If it still fails: your account may be out of **free GPU quota** — wait a few hours, or accept CPU (this notebook auto-switches to SD-Turbo 4-step mode).
4. Try a different network/hotspot — this network dropped external hosts all evening (codeload/OSM failed earlier).
5. Sync your system clock (TLS issues break gstatic handshakes even when the page loads).
6. Last resort: `Ctrl+F9` again after a `Runtime → Restart and run all`.

**Other issues:**
- No GPU attached → `Runtime → Change runtime type → GPU → Save`, then re-run.
- First download is ~4 GB — do not close the tab during Section 4.
- Images soft/blurry on CPU mode → expected; set `EP_SCENARIO_STEPS=8` next run for a quality/speed tradeoff.
- Rate limit/429 → wait, or set `EP_SCENARIO_MODEL` to an alternate repo before Section 4.

### Pushing to Hugging Face Hub (optional, for the live demo)
```python
!huggingface-cli login        # paste your token
!huggingface-cli upload <you>/earth-pulse-scenarios /content/earth-pulse/models/artifacts/scenarios
```